In [1]:
import pandas as pd
df = pd.read_csv('test_data.csv')

In [2]:
gamma = 0.9
reward_goal = 1.0
reward_step = -0.01

#up right down left
ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]

In [3]:
env = df['type'].to_numpy().reshape(6, 6)
env

array([[2, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0],
       [0, 1, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 3]])

In [11]:
def get_next_state(state, action):
    new_state = (state[0] + action[0], state[1] + action[1])
    if 0 <= new_state[0] <= 5 and 0 <= new_state[1] <= 5 and env[new_state] != 1:
        return new_state
    else:
        return state

In [22]:
import numpy as np
V = np.zeros((6, 6))

while True:
    delta = 0

    for i in range(0,6):
        for j in range(0,6):
            state = (i, j)
            if env[state] == 1 or env[state] == 3:
                continue

            q_values = []
            for k in range(len(ACTIONS)):
                q = 0
                for prob, action_index in [[0.8, k], [0.1, (k+1)%4], [0.1, (k-1)%4]]:
                    new_state = get_next_state(state, ACTIONS[action_index])
                    reward = reward_goal if env[new_state] == 3 else reward_step
                    q+=prob*(reward + gamma * V[new_state])
                q_values.append(q)
            
            best_q = max(q_values)
            delta = max(delta, best_q-V[state])
            V[state] = best_q

    if delta < 1e-8:
        break

In [23]:
V.round(2)

array([[0.23, 0.28, 0.33, 0.38, 0.44, 0.51],
       [0.27, 0.  , 0.39, 0.45, 0.  , 0.61],
       [0.32, 0.37, 0.45, 0.53, 0.62, 0.71],
       [0.37, 0.32, 0.  , 0.62, 0.71, 0.83],
       [0.44, 0.  , 0.62, 0.71, 0.  , 0.97],
       [0.51, 0.61, 0.71, 0.83, 0.97, 0.  ]])

In [ ]:
policy = np.zeros((6,6))

for i in range(0,6):
    for j in range(0,6):
        state = (i, j)
        if env[state] == 1 or env[state] == 3:
            continue

        q_values = []
        for k in range(len(ACTIONS)):
            q = 0
            for prob, action_index in [[0.8, k], [0.1, (k+1)%4], [0.1, (k-1)%4]]:
                new_state = get_next_state(state, ACTIONS[action_index])
                reward = reward_goal if env[new_state] == 3 else reward_step
                q+=prob*(reward + gamma * V[new_state])
            q_values.append(q)
        
        best_action = np.argmax(q_values)
        policy[state] = best_action

policy = 

array([[1., 1., 1., 2., 1., 2.],
       [2., 0., 2., 2., 0., 2.],
       [2., 1., 1., 1., 1., 2.],
       [2., 0., 0., 2., 1., 2.],
       [2., 0., 1., 2., 0., 2.],
       [1., 1., 1., 1., 1., 0.]])